# Phase 3 Colab Setup — FSRCNN and IMDN

Run this short notebook in a **separate Colab GPU session** while the NEDI Step 6 job continues. It only validates the Phase 3 environment and storage conventions; it does not train a model or alter Phase 2 outputs.

## 1. Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → T4 GPU** before running the next cells.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/divinesta/SuperResolution-Comparative-Analysis.git'
REPO_ROOT = Path('/content/SuperResolution-Comparative-Analysis')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

In [ ]:
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import platform
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Select a T4 GPU runtime and reconnect.')

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
from app.deep_learning.config import DeepLearningModelConfig

DATA_ROOT = Path('/content/drive/MyDrive/FYP_SR_Data')
CHECKPOINT_ROOT = DATA_ROOT / 'checkpoints'
OUTPUT_ROOT = DATA_ROOT / 'results' / 'phase3'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configs = [
    DeepLearningModelConfig(model, scale, CHECKPOINT_ROOT)
    for model in ('fsrcnn', 'imdn')
    for scale in (2, 3, 4)
]
for config in configs:
    status = 'found' if config.checkpoint_path.is_file() else 'not downloaded yet'
    print(f'{config.model.upper()} x{config.scale}: {status} — {config.checkpoint_path}')

## Setup completion test

This setup is successful when the GPU details print and all six scale-specific checkpoint paths are listed. Missing checkpoints are expected at this stage; they will be added only after their architecture and provenance are verified.